# 01 — Data exploration

Phase 1 is finished; this notebook is where you look at what you collected and write down what's wrong with it. Every section ends with **a decision or a number you carry forward** — nothing here is decoration.

What comes out of this notebook:

| Section | Produces |
|---|---|
| 1 Volume over time | Sanity check on the time split |
| 2 Axis structure | The table that goes in the README |
| 3 Multi-label shape | Confirms sigmoid + BCE is the right setup |
| 4 Imbalance | The `pos_weight` justification for Phase 3 |
| 5 Token length | **The 512-truncation number Phase 3 needs** |
| 6 Domain shift | Whether the time split is fair |
| 7 Bot-command leak | Validates the stripping decision |
| 8 Near-synonyms | Interview material |
| 9 `impact` axis audit | **Whether you have 5 axes or 4** |

Sections 5 and 9 are the two that block later phases. The rest is write-up material.

Run from the repo root or from `notebooks/` — the setup cell handles both.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINDINGS = {}   # collected at the end into results/data_exploration.json


def load(name):
    df = pd.read_parquet(ROOT / 'data/processed' / (name + '.parquet'))
    df['y'] = df['y'].apply(list)
    return df


tr, va, te = load('train'), load('val'), load('test')
tr['split'], va['split'], te['split'] = 'train', 'val', 'test'
full = pd.concat([tr, va, te], ignore_index=True)

stats = json.load(open(ROOT / 'results/label_stats.json'))
AXES = stats['kept_axes']
LABEL_AXIS = {lab: ax for ax, d in AXES.items() for lab in d['labels']}

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('{:,} labeled issues across {} axes'.format(len(full), len(AXES)))
for a, d in AXES.items():
    print('  {:10} {:3} labels  {:7,} issues'.format(a, d['n_labels'], d['n_issues']))

## 1 — Volume over time

The repo grew enormously between 2010 and today, which is why the 80/10/10 split by *count* gives train 14 years and test 14 months. That is the correct choice — you want to test on recent issues, because that's what deployment looks like — but you should be able to say out loud that the split is wildly uneven in time.

**Look for:** whether any year is anomalously large or small (a mass-import or a bot spree would show here).

In [ ]:
per_year = full.groupby(full['created_at'].dt.year).size()

fig, ax = plt.subplots()
colors = ['#4C72B0' if y < 2024 else '#DD8452' for y in per_year.index]
ax.bar(per_year.index, per_year.values, color=colors)
ax.set_title('Labeled issues created per year (orange = val/test era)')
ax.set_xlabel('year')
ax.set_ylabel('issues')
plt.show()

for name, part in [('train', tr), ('val', va), ('test', te)]:
    span = (part['created_at'].max() - part['created_at'].min()).days
    print('{:6} {:6,} issues over {:5,} days  ({} -> {})'.format(
        name, len(part), span,
        part['created_at'].min().date(), part['created_at'].max().date()))

FINDINGS['issues_per_year'] = {int(k): int(v) for k, v in per_year.items()}

## 2 — Axis structure

This is the table that goes near the top of the README. The asymmetry matters: `area` has 77 labels and `priority` has 4. Macro-F1 averages per-label F1 equally, so `area` will score low partly *because* it has 77 labels, many near the 100-support floor — not only because it's harder.

**Decision:** report micro-F1 beside macro-F1, and put the label count in the table so nobody reads a low `area` score as a pure difficulty signal.

In [ ]:
rows = []
for a, d in AXES.items():
    sup = sorted(d['labels'].values(), reverse=True)
    rows.append({
        'axis': a,
        'labels': d['n_labels'],
        'issues': d['n_issues'],
        'coverage': d['n_issues'] / len(full),
        'largest': sup[0],
        'smallest': sup[-1],
        'imbalance': sup[0] / sup[-1],
    })

axis_tbl = pd.DataFrame(rows).sort_values('issues', ascending=False)
display(axis_tbl.style.format({
    'issues': '{:,}', 'coverage': '{:.1%}',
    'largest': '{:,}', 'smallest': '{:,}', 'imbalance': '{:.0f}x'}))

fig, ax = plt.subplots()
ax.bar(axis_tbl['axis'], axis_tbl['issues'], color='#4C72B0')
ax.set_yscale('log')
ax.set_title('Issues per axis (log scale)')
ax.set_ylabel('issues with >=1 label on this axis')
plt.show()

FINDINGS['axis_table'] = axis_tbl.to_dict('records')

## 2b — Coverage, and is `priority` a representative subset?

Section 2 measured imbalance *within* axes. There is a second and more dangerous kind: **coverage**. `priority` is attached to 4,567 of 51,294 issues — 91% of your data has no priority label at all.

That is not the same as 91% being low-priority. Nobody assessed them. If you train BCE on the raw label vector, ~46,000 unassessed issues become confident negatives for every priority label, and the model learns that almost nothing is urgent. The fix is per-axis masking — train and score each axis only on issues labeled on that axis — which is a Phase 3 change.

But masking only helps if the labeled subset is *representative*. If rust prioritizes issues that were already escalated — noisy, long-lived, already argued over — then a model trained on that subset learns 'was this escalated' rather than 'how urgent is this', and would be useless on a freshly filed ticket. That is the failure mode that would genuinely justify dropping the axis.

The cells below test it. They need `n_comments` and `closed_at`, which aren't in the parquet files, so they re-read `data/raw` (about 30 seconds).

In [ ]:
import gzip

cov = pd.DataFrame([
    {'axis': a, 'labeled': d['n_issues'],
     'unlabeled': len(full) - d['n_issues'],
     'coverage': d['n_issues'] / len(full)}
    for a, d in AXES.items()]).sort_values('coverage', ascending=False)
display(cov.style.format({'labeled': '{:,}', 'unlabeled': '{:,}', 'coverage': '{:.1%}'}))
print('unlabeled = issues that become fabricated negatives without masking\n')

# Pull metadata the parquet files don't carry.
meta = {}
for p in sorted((ROOT / 'data/raw').glob('*.jsonl.gz')):
    with gzip.open(p, 'rt', encoding='utf-8') as fh:
        for line in fh:
            it = json.loads(line)
            meta[it['number']] = (it.get('comments', 0), it.get('closed_at'))

full['n_comments'] = full['number'].map(lambda n: meta.get(n, (0, None))[0])
full['closed_at'] = pd.to_datetime(
    full['number'].map(lambda n: meta.get(n, (0, None))[1]), utc=True, errors='coerce')
full['days_to_close'] = (full['closed_at'] - full['created_at']).dt.days
full['is_closed'] = full['closed_at'].notna()

AXIS_MASK = {a: full['y'].apply(lambda ys: any(LABEL_AXIS[y] == a for y in ys))
             for a in AXES}
print('metadata attached for {:,} issues'.format(len(meta)))

In [ ]:
has = AXIS_MASK['priority']
prio, rest = full[has], full[~has]

rows = []
for name, g in [('prioritized', prio), ('everything else', rest)]:
    rows.append({
        'group': name,
        'n': len(g),
        'median comments': g['n_comments'].median(),
        'mean comments': g['n_comments'].mean(),
        'closed': g['is_closed'].mean(),
        'median days to close': g['days_to_close'].median(),
        'mean labels': g['y'].str.len().mean(),
    })
cmp = pd.DataFrame(rows)
display(cmp.style.format({
    'n': '{:,}', 'median comments': '{:.1f}', 'mean comments': '{:.1f}',
    'closed': '{:.1%}', 'median days to close': '{:.0f}', 'mean labels': '{:.2f}'}))

ratio = (cmp.loc[0, 'mean comments'] + 1e-9) / (cmp.loc[1, 'mean comments'] + 1e-9)
print('prioritized issues carry {:.1f}x the comments of the rest'.format(ratio))
FINDINGS['priority_comment_ratio'] = round(float(ratio), 2)

fig, ax2 = plt.subplots(1, 2, figsize=(11, 4))
bins = np.arange(0, 31)
ax2[0].hist(rest['n_comments'].clip(0, 30), bins=bins, density=True,
            alpha=0.6, label='rest', color='#4C72B0')
ax2[0].hist(prio['n_comments'].clip(0, 30), bins=bins, density=True,
            alpha=0.6, label='prioritized', color='#DD8452')
ax2[0].set_title('Comment count (clipped at 30)')
ax2[0].set_xlabel('comments')
ax2[0].legend()

share = full.assign(p=has.values).groupby(
    full['created_at'].dt.year)['p'].mean()
ax2[1].plot(share.index, share.values, marker='o', color='#DD8452')
ax2[1].set_title('Share of issues given a priority label, by year')
ax2[1].set_ylabel('share')
plt.tight_layout()
plt.show()

In [ ]:
# Which labels are over-represented among prioritized issues?
# lift = P(label | prioritized) / P(label overall). Near 1.0 means
# prioritization cuts across the taxonomy. 5x+ on a few labels means the
# prioritized subset is really 'issues of type X', and a priority model would
# partly be a type detector wearing a different name.
all_counts = Counter(l for ys in full['y'] for l in ys)
prio_counts = Counter(l for ys in prio['y'] for l in ys)

lift = []
for lab, n in all_counts.items():
    if LABEL_AXIS[lab] == 'priority' or n < 200:
        continue
    p_all = n / len(full)
    p_prio = prio_counts.get(lab, 0) / len(prio)
    lift.append({'label': lab, 'axis': LABEL_AXIS[lab],
                 'p_overall': p_all, 'p_prioritized': p_prio,
                 'lift': (p_prio + 1e-9) / (p_all + 1e-9)})

lf = pd.DataFrame(lift).sort_values('lift', ascending=False)
fmt = {'p_overall': '{:.1%}', 'p_prioritized': '{:.1%}', 'lift': '{:.2f}x'}
print('Most over-represented among prioritized issues:')
display(lf.head(10).style.format(fmt))
print('Most under-represented:')
display(lf.tail(6).style.format(fmt))

print('\nmedian lift across labels: {:.2f}x'.format(lf['lift'].median()))
FINDINGS['priority_median_lift'] = round(float(lf['lift'].median()), 3)

### Reading it

**Representative enough to keep, with a caveat** — comment ratio under ~2x, lift mostly between 0.5x and 2x, prioritization share reasonably flat across years. Keep the axis, mask it in training, and note that priority is assigned to a triaged subset.

**Genuinely a different population** — prioritized issues carry several times the comments, a handful of labels show 5x+ lift, or prioritization barely appears before some year. The model would be learning 'this issue got attention', not 'this issue is urgent'. Still report it, but state plainly that the priority result is measured on escalated issues and says nothing about a freshly filed ticket.

**The second panel matters as much as the first.** If prioritization only became common recently, most of your 2010–2024 training data predates the practice and the axis is thinner than 4,567 suggests.

Dropping the axis is the wrong move under either outcome. A measured, caveated negative result on the operationally most valuable label *is* the finding. Silently removing it is the failure the build plan warns about — reporting only the good axes.

### What I found

*(fill in)*

## 3 — How multi-label is this really?

If most issues carried exactly one label, a softmax classifier would be fine and the whole multi-label setup would be overkill. Check before committing to sigmoid + `BCEWithLogitsLoss`.

**Look for:** the mean labels-per-issue, and specifically whether any single axis regularly carries more than one label at once (an issue tagged both `A-diagnostics` and `A-parser`). If axes are single-label internally, that's worth knowing — it means per-axis softmax is defensible and you chose sigmoid for a reason you should be able to state.

In [ ]:
n_labels = full['y'].str.len()

fig, ax = plt.subplots()
ax.hist(n_labels, bins=range(1, n_labels.max() + 2), color='#4C72B0', align='left')
ax.set_title('Labels per issue')
ax.set_xlabel('kept labels')
ax.set_ylabel('issues')
plt.show()

print('mean {:.2f}, median {:.0f}, max {}'.format(
    n_labels.mean(), n_labels.median(), n_labels.max()))
print()
print('Multi-label WITHIN a single axis:')
for a in AXES:
    per = full['y'].apply(lambda ys: sum(1 for y in ys if LABEL_AXIS[y] == a))
    has = per[per > 0]
    print('  {:10} mean {:.2f}   {:5.1f}% of tagged issues have 2+'.format(
        a, has.mean(), 100 * (has > 1).mean()))

FINDINGS['labels_per_issue_mean'] = round(float(n_labels.mean()), 3)

## 4 — Imbalance

The `imbalance` column above is the ratio of the most common label to the rarest within each axis. This is the evidence behind the Phase 3 `pos_weight` decision — the `{{approach}}` placeholder on the resume.

**Look for:** the head/tail shape. If the top label in an axis covers most of that axis's issues, a model that predicts only the head label will look deceptively reasonable on micro-F1, which is exactly why the majority-class floor in Phase 2 exists.

In [ ]:
counts = Counter(l for ys in full['y'] for l in ys)
top20 = pd.Series(dict(counts.most_common(20))).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top20.index, top20.values, color='#4C72B0')
ax.set_title('20 most common labels')
ax.set_xlabel('issues')
plt.tight_layout()
plt.show()

print('Head share within each axis:')
for a, d in AXES.items():
    sup = sorted(d['labels'].items(), key=lambda kv: -kv[1])
    head, n = sup[0]
    print('  {:10} {:28} {:5.1f}% of axis'.format(a, head, 100 * n / d['n_issues']))

## 5 — Token length  *(blocks Phase 3)*

DistilBERT truncates at 512 tokens. You need to know what fraction of issues you're cutting off, and — more importantly — whether the truncated ones are systematically different.

This needs the real tokenizer, not a word-count approximation: rust issues are full of code, paths, and backtraces, which subword tokenizers explode far beyond word count. An approximation here would give you a confidently wrong truncation rate.

If `transformers` isn't installed yet, install it — Phase 3 needs it regardless.

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    HAVE_TOK = True
except Exception as exc:
    HAVE_TOK = False
    print('tokenizer unavailable:', type(exc).__name__, exc)
    print('\n    pip install transformers\n')

if HAVE_TOK:
    sample = full.sample(min(4000, len(full)), random_state=0)
    lens = np.array([len(tok.encode(t, truncation=False, add_special_tokens=True))
                     for t in sample['text']])

    fig, ax = plt.subplots()
    ax.hist(np.clip(lens, 0, 2000), bins=80, color='#4C72B0')
    ax.axvline(512, color='#C44E52', lw=2, label='512 (DistilBERT limit)')
    ax.set_title('Token length of cleaned text (clipped at 2000 for display)')
    ax.set_xlabel('tokens')
    ax.legend()
    plt.show()

    over = (lens > 512).mean()
    print('median {:.0f} tokens, p90 {:.0f}, p99 {:.0f}'.format(
        np.median(lens), np.percentile(lens, 90), np.percentile(lens, 99)))
    print('{:.1%} of issues exceed 512 tokens'.format(over))
    FINDINGS['pct_over_512'] = round(float(over), 4)
    FINDINGS['median_tokens'] = int(np.median(lens))

Truncation isn't uniform. Long issues are long for a reason — huge backtraces, big reproductions — and those correlate with certain labels. Check which labels are over-represented among the truncated.

**Carry forward to Phase 4:** if a label's issues are mostly truncated, poor performance on it may be a context-window problem rather than an unlearnable-signal problem. Those are very different findings and you don't want to confuse them.

In [ ]:
if HAVE_TOK:
    sample = sample.copy()
    sample['truncated'] = lens > 512
    base = sample['truncated'].mean()

    rows = []
    for lab in counts:
        m = sample['y'].apply(lambda ys: lab in ys)
        if m.sum() >= 30:
            rows.append({'label': lab, 'axis': LABEL_AXIS[lab],
                         'n': int(m.sum()),
                         'trunc_rate': float(sample.loc[m, 'truncated'].mean())})

    trunc = pd.DataFrame(rows).sort_values('trunc_rate', ascending=False)
    print('overall truncation rate {:.1%}\n'.format(base))
    print('Most truncated:')
    display(trunc.head(10).style.format({'trunc_rate': '{:.1%}'}))
    print('Least truncated:')
    display(trunc.tail(5).style.format({'trunc_rate': '{:.1%}'}))

## 6 — Domain shift across the split

You train on 2010–2024 and test on 2025–2026. If a label barely existed before 2024, the model has almost no training signal for it but will be graded on it. If a label was retired, the reverse.

**Look for:** any label whose share of issues changes sharply at the train/test boundary. A few are normal. Many means the time split is doing more work than you intended, and it belongs in the write-up as a limitation.

In [ ]:
top8 = [l for l, _ in counts.most_common(8)]
yearly = full.groupby(full['created_at'].dt.year).size()

fig, ax = plt.subplots(figsize=(10, 5))
for lab in top8:
    share = (full[full['y'].apply(lambda ys: lab in ys)]
             .groupby(full['created_at'].dt.year).size() / yearly)
    ax.plot(share.index, share.values, marker='o', ms=3, label=lab)
ax.axvline(2024.4, color='#C44E52', ls='--', lw=1.5, label='train/val boundary')
ax.set_title('Label prevalence over time (share of that year\'s issues)')
ax.set_ylabel('share')
ax.legend(fontsize=8, ncol=2)
plt.show()

print('Largest train -> test prevalence shifts:')
shift = []
for lab in counts:
    a = tr['y'].apply(lambda ys: lab in ys).mean()
    b = te['y'].apply(lambda ys: lab in ys).mean()
    if a > 0.002 or b > 0.002:
        shift.append({'label': lab, 'train': a, 'test': b, 'ratio': (b + 1e-6) / (a + 1e-6)})
sh = pd.DataFrame(shift).sort_values('ratio')
display(pd.concat([sh.head(6), sh.tail(6)]).style.format(
    {'train': '{:.2%}', 'test': '{:.2%}', 'ratio': '{:.2f}x'}))

## 7 — The bot-command leak

`prepare.py` strips `@rustbot label ...` from issue bodies, because rustbot reads the command and applies the label — the text *causes* the target. Left in, the model learns a regex.

The search API measured 8.0% within `C-bug` (1,681 / 21,037). Your pipeline reports 3.7% overall, but that denominator includes unlabeled issues and axes where commands are rare.

**The check:** does the per-label rate for `C-bug` land near 8%? If it's far below, `BOT_CMD` is missing command variants and some leakage is still in your training text.

In [ ]:
by_label = stats.get('bot_cmd_rate_by_label', {})
if not by_label:
    print('re-run data/prepare.py — per-label rates not in results/label_stats.json')
else:
    print('overall rate: {:.1%}'.format(stats['bot_cmd_rate_overall']))
    print('C-bug rate:   {:.1%}   (search API said 8.0%)'.format(
        by_label.get('C-bug', float('nan'))))
    print()
    s = pd.Series(by_label).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(s.index[::-1], s.values[::-1], color='#C44E52')
    ax.set_title('Share of issues whose body carried a rustbot label command')
    ax.set_xlabel('rate')
    plt.tight_layout()
    plt.show()

# Does having carried a command correlate with anything else? If commanded
# issues are systematically different, stripping fixes the text but leaves a
# population difference worth reporting.
cmd = full[full['had_bot_cmd']]
print('commanded issues: {:,}  ({:.1%})'.format(len(cmd), len(cmd) / len(full)))
print('  mean labels  {:.2f}  vs  {:.2f} for the rest'.format(
    cmd['y'].str.len().mean(), full[~full['had_bot_cmd']]['y'].str.len().mean()))

## 8 — Near-synonyms

This one is manual and it's the section that produces interview material. Read the label list for each axis and note any pair you would have struggled to choose between while triaging.

You did exactly this by hand at Endurance against a taxonomy you wrote. That experience is the reason this section is worth your time and not just a formality — the observation you make here is the same observation that goes in the Endurance ambiguity notes.

**Write your pairs into the markdown cell below as you find them.**

In [ ]:
for a, d in AXES.items():
    labs = sorted(d['labels'].items(), key=lambda kv: -kv[1])
    print('\n=== {} ({} labels) ==='.format(a.upper(), len(labs)))
    for i in range(0, len(labs), 3):
        print('   ' + '  '.join(
            '{:26}{:>6,}'.format(l, n) for l, n in labs[i:i + 3]))

### Ambiguous pairs I found

*(fill in)*

- 
- 
- 

**Which axis would I least want to apply by hand, and why?**

*(fill in)*

## 9 — The `impact` axis audit  *(blocks the axis count)*

Rust's `triagebot.toml` has autolabel rules that fire on issue creation based on which template was used. File through the ICE template and `I-ICE` is applied automatically — no human judgment, and no command text to strip.

If that's what's happening, `impact` is a leak dressed as an axis, and you have **four** axes rather than five.

**The test is timing, not actor.** Rustbot applies labels for humans too, so the actor tells you nothing. But an autolabel fires within seconds of issue creation, while a human takes minutes to days. That separation is clean and cheap to measure.

This makes ~40 authenticated API calls.

In [ ]:
import os, time, random
import requests
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
REPO = json.load(open(ROOT / 'results/label_stats.json'))['repo']
S = requests.Session()
S.headers.update({
    'Authorization': 'Bearer ' + os.environ['GITHUB_TOKEN'],
    'Accept': 'application/vnd.github+json',
})


def label_delays(issue_number):
    """Seconds between issue creation and each label application."""
    r = S.get('https://api.github.com/repos/{}/issues/{}/timeline'.format(
        REPO, issue_number), params={'per_page': 100}, timeout=30)
    time.sleep(0.4)
    if not r.ok:
        return []
    events = r.json()
    created = pd.Timestamp(
        full.loc[full['number'] == issue_number, 'created_at'].iloc[0])
    out = []
    for e in events:
        if e.get('event') == 'labeled' and e.get('created_at'):
            out.append({
                'label': (e.get('label') or {}).get('name'),
                'actor': (e.get('actor') or {}).get('login'),
                'delay_s': (pd.Timestamp(e['created_at']) - created).total_seconds(),
            })
    return out


random.seed(0)
AUDIT_LABELS = ['I-ICE', 'I-unsound', 'C-bug']   # C-bug is the control
rows = []
for lab in AUDIT_LABELS:
    pool = full[full['y'].apply(lambda ys: lab in ys)]['number'].tolist()
    for n in random.sample(pool, min(15, len(pool))):
        for ev in label_delays(n):
            if ev['label'] == lab:
                rows.append({'target': lab, **ev, 'number': n})

audit = pd.DataFrame(rows)
print('{} label events across {} issues\n'.format(len(audit), audit['number'].nunique()))
for lab, g in audit.groupby('target'):
    fast = (g['delay_s'] < 30).mean()
    print('{:12} median delay {:>10.0f}s   {:5.1%} applied within 30s   actors: {}'.format(
        lab, g['delay_s'].median(), fast, ', '.join(sorted(set(g['actor']))[:3])))

FINDINGS['impact_audit'] = audit.groupby('target')['delay_s'].median().to_dict()

### Reading the result

- **`I-ICE` median delay near zero and `C-bug` in minutes/hours** → `I-ICE` is template-applied. Drop `impact` from `axes.yaml` with the reason recorded, and the project has four axes.
- **Both delayed similarly** → humans are applying `I-ICE` too. Keep the axis, and note in the write-up that you checked.
- **Mixed** → some `I-` labels are automatic and others aren't. Exclude the automatic ones individually rather than the whole axis.

Whatever you find, write it down. "I checked whether my labels were bot-generated and here's how" is a stronger answer than any F1 score.

### What I found

*(fill in)*

## 10 — Write it down

Saves the numbers to `results/data_exploration.json` so nothing lives only in this notebook's output cells. Then write `results/DATA_NOTES.md` — one page, in prose, answering:

1. How many issues, over what range, after filtering PRs
2. Which axes survived and which didn't, with numbers
3. How you determined which labels were bot-applied
4. What you stripped from the text and why
5. The worst imbalance you found
6. Anything that made you uneasy

That page is the raw material for sections 2 and 6 of the write-up.

In [ ]:
out = ROOT / 'results/data_exploration.json'
json.dump(FINDINGS, open(out, 'w'), indent=2, default=str)
print('wrote', out)
for k, v in FINDINGS.items():
    if not isinstance(v, (dict, list)):
        print('  {:24} {}'.format(k, v))